# ECONOMETRIA II

In [ ]:
#pip install statmodels.api

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from linearmodels.iv import IV2SLS as IV

In [ ]:
import wooldridge

In [ ]:
df = wooldridge.data('mroz')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
#df = df[df['lwage'].notna()]
df = df.dropna()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

## MC2E

In [ ]:
Y1 = df['lwage']
Y2 = df['educ']
X = df[['exper' , 'expersq']] #variables sin endogeneidad
Z = df[['motheduc' , 'fatheduc']]

In [ ]:
exog_inst = [X , Z]
reg1e = pd.concat(exog_inst , axis=1)
exog_MCO = [ X , Y2]
regMCO =  pd.concat(exog_MCO , axis=1)

In [ ]:
reg1e

In [ ]:
model_1e = sm.OLS(df.educ, sm.add_constant(reg1e))

In [ ]:
result_1e = model_1e.fit()

In [ ]:
print(result_1e.summary())

la relevancia de las variables instrumentales queda comprobada por la significancia estadistica de los parámetros, que deden ser distintos de 0. Observar también que el 0 no se encuentra comprendido en los intervalos correspondientes a cada parametro estimado

también puede desarrollarse una prueba de significancia conjunta a traves de un test F:

In [ ]:
test_F = result_1e.f_test('fatheduc = 0 , motheduc = 0 ' )

print(test_F)

In [ ]:
df['educ_hat'] = result_1e.predict()

In [ ]:
reg2e = pd.concat([X , df.educ_hat] , axis=1 )

In [ ]:
reg2e.head()

Ahora, construimos el modelo tomando como variable dependiente la variable dependiente de la ecuación original(y1) y como variables explicativas las variables explicativas exógenas de la ecuaciónoriginal (las x, siguiendo la notación inicial) y la variable estimada en la primera etapa(¡ ˆy2 y no y2!).


In [ ]:
model_2e = sm.OLS(Y1, sm.add_constant(reg2e))

result_2e = model_2e.fit()

print(result_2e.summary(alpha=0.1))

Esta forma manual de realizar el procedimiento de MCO en 2 etapas es solo explicativa porque contiene ee erroreos. Es necesario utilizar la función contenida en el paquete para que calcule automaticamente las 2 etapas conteniendo la correccion de los errores tambien

In [ ]:
MC2E = IV(dependent= Y1 , exog = sm.add_constant(X) , endog = Y2, instruments = Z)

In [ ]:
resultado = MC2E.fit()

In [ ]:
print(resultado.summary)

In [ ]:
from linearmodels.iv import compare

In [ ]:
result_MCO = IV(dependent = Y1, exog = sm.add_constant(regMCO) , endog = None , instruments = None).fit()

In [ ]:
result_man = IV(dependent = Y1, exog = sm.add_constant(reg2e) , endog = None , instruments = None).fit()

In [ ]:
result_MC2E = IV(dependent = Y1, exog = sm.add_constant(X) , endog = Y2 , instruments = Z).fit()

In [ ]:
#resultados_comp = compare({'MCO':result_MCO , 'MC2E manual' : result_man , 'MC2E' : result_MC2E}, stars = True , precision = 'std_errors')

In [ ]:
resultados_comp = compare({'MCO':result_MCO , 'MC2E' : result_MC2E}, stars = True , precision = 'std_errors')

In [ ]:
print(resultados_comp)

In [ ]:
help(resultados_comp)

Relevancia del instrumento

* Cuanto más relevante es el instrumento hay más información disponible para estimar por variables instrumentales.

* Instrumentos que explican una pequeña fracción de la variación de 𝑥 son denominados instrumentos débiles.

* En el caso extremo en el que el instrumento es irrelevante (𝐶𝑜𝑣(𝑥;𝑧)=0), el estimador de MC2E no es consistente:
𝛽̂1𝑀𝐶2𝐸=𝐶𝑜𝑣𝑛(𝑧,𝑦)/𝐶𝑜𝑣𝑛(𝑥,𝑧) 𝑃→ 𝐶𝑜𝑣(𝑧,𝑦)/𝐶𝑜𝑣(𝑥,𝑧)
o El denominador es cero.

* El estadístico 𝑭 de la primera etapa (forma reducida) provee una medida de la información contenida en los instrumentos. Medida de la relevancia de los instrumentos.

* Cuanto mayor es la información, mayor es el valor esperado del estadístico 𝐹.

* Regla práctica: si 𝐹<10, es evidencia a favor de la presencia de instrumentos débiles.

Debe notarse también que una medida del 𝑅2 en el contexto de IV no es “el porcentaje de la variación explicada” como en MCO. En presencia de correlación entre x y u no es posible descomponer la varianza de y en dos partes independientes (SSE y SSR) y el 𝑅2 no tiene una interpretación natural.